## FragranceDNA ##

This notebook explores a dataset of ~24,000 fragrances scraped from Fragrantica, one of the largest fragrance communities on the web. Each entry includes the fragrance's top, middle, and base notes, brand, gender classification, and user ratings.

The goal is to build a fragrance recommender system that goes beyond simple note matching, using several methods from the domain of linear algebra and machine learning to find more meaningful similarities between fragrances and discover underlying scent families.

I will start by cleaning the dataset, then build a note matrix, and test various similarity metrics before finishing with nonnegative matrix factorization, a technique that can independently recover the four fragrance families described by Michael Edwards' fragrance wheel and give users new to fragrance a vocabulary to seek out fragrances on their own.
 

In [2]:
# First import the dataset
import pandas as pd

df = pd.read_csv('fra_cleaned.csv', encoding='latin-1', sep=';')
print(df.shape) # prints the rows and columns of the dataset
print(df.head()) # displays first five entries

(24063, 18)
                                                 url  \
0  https://www.fragrantica.com/perfume/xerjoff/ac...   
1  https://www.fragrantica.com/perfume/jean-paul-...   
2  https://www.fragrantica.com/perfume/jean-paul-...   
3  https://www.fragrantica.com/perfume/bruno-bana...   
4  https://www.fragrantica.com/perfume/jean-paul-...   

                          Perfume               Brand  Country  Gender  \
0  accento-overdose-pride-edition             xerjoff    Italy  unisex   
1            classique-pride-2024  jean-paul-gaultier   France   women   
2            classique-pride-2023  jean-paul-gaultier   France  unisex   
3               pride-edition-man        bruno-banani  Germany     men   
4         le-male-pride-collector  jean-paul-gaultier   France     men   

  Rating Value  Rating Count    Year  \
0         1,42           201  2022.0   
1         1,86            70  2024.0   
2         1,91           285  2023.0   
3         1,92            59  2019.0   
4     

## Clean the dataset - Module 4 enhancement ##

Before writing the cleaning code, I created a unique notes list in Excel using a pivot table to sort by frequency.  Working outside the notebook allowed me to visually scan for misspellings, variants, and synonyms.  This would have been tedious to do programatically and requires domain-specific knowledge to avoid combining terms that represent two distinct things, or catching regional variants (agarwood/oud)

In [7]:
import pandas as pd

df = pd.read_csv('fra_cleaned.csv', encoding='latin-1', sep=';')

# note corrections map
corrections = {
    'oud': 'agarwood (oud)',
    'agarwood': 'agarwood (oud)',
    'agave nectar': 'agave',
    'ambrette': 'ambrette (musk mallow)',
    'apple tree blossom': 'apple blossom',
    'basil leaf': 'basil',
    'camelia': 'camellia',
    'canada balsam': 'canadian balsam',
    'carambola': 'carambola (star fruit)',
    'cardamon': 'cardamom',
    'cashmirwood': 'cashmere wood',
    'cinammon': 'cinnamon',
    'citruses': 'citrus',
    'civetta': 'civet',
    'cloves': 'clove',
    'coton candy': 'cotton candy',
    'cupuacu': 'cupuassu',
    'darjeeling black tea': 'darjeeling tea',
    'fruity': 'fruity notes',
    'graperfuit': 'grapefruit',
    'green grass': 'grass',
    'green accord': 'green notes',
    'juniper berry': 'juniper berries',
    'karo-korund': 'karo-karounde',
    'lily of the valley': 'lily-of-the-valley',
    'lime (linden) blossom': 'lime (linden blossom)',
    'marshamallow': 'marshmallow',
    'massoia wood': 'massoia',
    'myrhh': 'myrrh',
    'orris': 'orris root',
    'sandalowood': 'sandalwood',
    'tahitian tiare flower': 'tiare flower',
    'tyger lily': 'tiger lily',
    'vanille': 'vanilla',
    'vanila': 'vanilla',
    'vetyver': 'vetiver',
    'white wood': 'white woods',
    'wisteria flower': 'wisteria',
    'woodsy notes': 'woody notes',
    'ylang ylang': 'ylang-ylang',
}

def fix_notes_cell(cell):
    if pd.isna(cell):
        return cell
    notes = [n.strip().lower() for n in cell.split(',')]
    return ', '.join(corrections.get(n, n) for n in notes)

for col in ['Top', 'Middle', 'Base']:
    df[col] = df[col].apply(fix_notes_cell)

# spot check: test for 0 instances of 'vetyver'
for col in ['Top', 'Middle', 'Base']:
    n = df[col].str.contains('vetyver', case=False, na=False).sum()
    print(f"{col}: {n} instances of 'vetyver'")

Top: 0 instances of 'vetyver'
Middle: 0 instances of 'vetyver'
Base: 0 instances of 'vetyver'


## Get all notes ##

The next step is to build the fragrance matrix - First, we will combine all the notes from the top, middle, and base notes columns into one 'set' per fragrance.  Not all fragrances in the dataset distinguish between top/middle/base notes, but most notes do inherently belond to one category by their chemical compositions, so this was a trade-off.

This gives each fragrance a list like ['vanilla', 'amber', 'vetiver']

In [8]:
def get_all_notes(row):
    notes = []
    for col in ['Top', 'Middle', 'Base']:
        if pd.notna(row[col]):
            notes.extend([n.strip().lower() for n in row[col].split(',')])
    return notes

df['all_notes'] = df.apply(get_all_notes, axis=1)

print(df[['Perfume', 'all_notes']].head()) # Prints first 5 fragrances with note lists

                          Perfume  \
0  accento-overdose-pride-edition   
1            classique-pride-2024   
2            classique-pride-2023   
3               pride-edition-man   
4         le-male-pride-collector   

                                           all_notes  
0  [fruity notes, aldehydes, green notes, bulgari...  
1  [yuzu, citrus, orange blossom, neroli, musk, b...  
2  [blood orange, yuzu, neroli, orange blossom, m...  
3  [guarana, grapefruit, red apple, walnut, laven...  
4  [mint, lavender, cardamom, artemisia, bergamot...  


## How many total notes? ##

This counts the unique notes in the dataset.

In [9]:
unique_notes = set()
for notes_list in df['all_notes']:
    unique_notes.update(notes_list)

print(f"Unique notes: {len(unique_notes)}")
print(f"\nFirst 10 (alphabetically): {sorted(unique_notes)[:10]}")

Unique notes: 1633

First 10 (alphabetically): ['absinthe', 'acai berry', 'accord eudora®', 'acerola', 'acerola blossom', 'acetylfuran', 'acácia', 'african freesia petals', 'african geranium', 'african ginger']


## Building the matrix - Module 4 enhancement ##

With our notes listed, we can populate the matrix using a dictionary data structure.  The dictionary/hash map approach is faster than scanning a sorted list - O(1) vs. O(n) time. A list lookup requires checking each element until a match is found, while a dictionary lookup finishes in constant time regardless of how many notes exist. 

In [54]:
import numpy as np

sorted_notes = sorted(unique_notes)

# Map each note to a column index using a dictionary for O(1) lookup
note_to_index = {note: i for i, note in enumerate(sorted_notes)}
note_matrix = np.zeros((len(df), len(note_to_index)), dtype=int)

for row_idx, notes_list in enumerate(df['all_notes']):
    for note in notes_list:
        col_idx = note_to_index[note]
        note_matrix[row_idx, col_idx] = 1

## Inspecting the matrix ##

With the matrix built, a few quick checks to see the shape of the matrix and another spot check of a known fragrance to ensure that its notes mapped correctly.

In [12]:
print(f"Matrix shape: {note_matrix.shape}")
print(f"  {note_matrix.shape[0]} fragrances x {note_matrix.shape[1]} notes\n")

# How many notes does each fragrance have on average?
notes_per_frag = note_matrix.sum(axis=1)
print(f"Notes per fragrance - min: {notes_per_frag.min()}, max: {notes_per_frag.max()}, avg: {notes_per_frag.mean():.1f}\n")

# How sparse is the matrix? (mostly zeros)
total_cells = note_matrix.shape[0] * note_matrix.shape[1]
filled_cells = note_matrix.sum()
print(f"Sparsity: {filled_cells}/{total_cells} cells filled ({filled_cells/total_cells*100:.2f}%)\n")

# spot check a known fragrance
idx = df[df['Perfume'].str.contains('shalimar', case=False)].index[0]
frag_notes = [note for note, col in note_to_index.items() if note_matrix[idx, col] == 1]
print(f"Spot check - {df.iloc[idx]['Perfume']}:")
print(f"  Notes: {frag_notes}")

Matrix shape: (24063, 1633)
  24063 fragrances x 1633 notes

Notes per fragrance - min: 1, max: 62, avg: 9.8

Sparsity: 236963/39294879 cells filled (0.60%)

Spot check - shalimar-souffle-d-oranger:
  Notes: ['bergamot', 'jasmine sambac', 'mandarin orange', 'neroli', 'orange blossom', 'petitgrain', 'sandalwood', 'vanilla']


## Selecting a fragrance ##

Now that we have more information on our matrix, we can find the index of a fragrance and use it for future similarity measures.

In [13]:
search = df[df['Perfume'].str.contains('dakar', case=False)][['Perfume', 'Brand']]
print(search)

     Perfume             Brand
4119   dakar  thera-cosmeticos


## Jaccard Similarity ##

The index of 'Dakar' is 4119 - We can assign this to target_idx and perform the Jaccard similarity test.

In [21]:
target_idx = 4119
target_row = note_matrix[target_idx]

# Jaccard: intersection / union for each fragrance
scores = []
for i in range(len(note_matrix)):
    if i == target_idx:
        continue
    intersection = np.sum((target_row == 1) & (note_matrix[i] == 1))
    union = np.sum((target_row == 1) | (note_matrix[i] == 1))
    score = intersection / union if union > 0 else 0
    scores.append((i, score))

# Sort by score descending, grab top 10
scores.sort(key=lambda x: x[1], reverse=True)

print(f"Recommendations for: {df.iloc[target_idx]['Perfume']} by {df.iloc[target_idx]['Brand']}\n")
print(f"Its notes: {df.iloc[target_idx]['all_notes']}\n")
print("Top 10 Jaccard matches:")
for rank, (idx, score) in enumerate(scores[:10], 1):
    print(f"  {rank}. {df.iloc[idx]['Perfume']} by {df.iloc[idx]['Brand']} — {score:.3f}")
    print(f"     Notes: {df.iloc[idx]['all_notes']}")

Recommendations for: dakar by thera-cosmeticos

Its notes: ['spicy notes', 'tobacco leaf', 'vanilla', 'tonka bean', 'tobacco blossom', 'cacao', 'woody notes', 'dried fruits']

Top 10 Jaccard matches:
  1. sweetobacco by in-the-box — 1.000
     Notes: ['tobacco leaf', 'spicy notes', 'vanilla', 'tobacco blossom', 'cacao', 'tonka bean', 'woody notes', 'dried fruits']
  2. efrate by nuancielo — 1.000
     Notes: ['spicy notes', 'tobacco leaf', 'vanilla', 'cacao', 'tobacco blossom', 'tonka bean', 'dried fruits', 'woody notes']
  3. essences-44 by nuancielo — 1.000
     Notes: ['tobacco leaf', 'spicy notes', 'cacao', 'vanilla', 'tonka bean', 'tobacco blossom', 'dried fruits', 'woody notes']
  4. tobacco-vanille by tom-ford — 1.000
     Notes: ['tobacco leaf', 'spicy notes', 'vanilla', 'cacao', 'tonka bean', 'tobacco blossom', 'dried fruits', 'woody notes']
  5. vanille-persuasive by lpdo — 0.778
     Notes: ['tobacco leaf', 'spicy notes', 'vanilla', 'tonka bean', 'tobacco blossom', 'cacao po

## END PART 1 OF ANALYSIS ##

Jaccard performed well as a simple test, matching Fragrantica's similar fragrances for Dakar.  But it has some limitations: literal notes overlap, and fragrances with different numbers of notes may be down weighted in this kind of analysis.

## Part 2 - Module 4 enhancement ##

## TF-IDF weighting ##

The TF-IDF (term frequency/inverse document frequency) addresses some of the issues with similarity metrics.  Musk, which appears in nearly half of all the fragrances, carries the same weight as a rarer note like chrysanthemum.

The 'term frequency' has already been built - the matrix contains a 1 wherever a fragrance contains a note, and nothing where the note is no present.  The weighting happens with the inverse document frequency: $$IDF(t) = \log\left(\frac{N}{df_t}\right)$$

Where $N$ is the total number of fragrances and $df_t$ is the number of fragrances containing note $t$.  Replacing the '1' values with the values produced by the IDF gives us a weighted matrix where each notes influence is correctly valued. The smaller the value, the more frequent the note is in the fragrance matrix.  

This weighted matrix also primes us to explore cosine similarity as a metric rather than just raw note overlap.

Below, we create the weighted matrix and display the lowest and highest IDF scores representing the most and least common notes.

In [28]:
from numpy.linalg import norm

N = note_matrix.shape[0]

# IDF 
df_t = note_matrix.sum(axis=0)  # how many fragrances contain each note
idf = np.log(N / df_t)

# TF-IDF matrix: multiply each row by the IDF vector
tfidf_matrix = note_matrix * idf

# pair each note with its IDF score
note_idf = list(zip(sorted_notes, idf))

note_idf_sorted = sorted(note_idf, key=lambda x: x[1])

print("10 lowest IDF scores (most common notes):")
for note, score in note_idf_sorted[:10]:
    print(f"  {note}: {score:.2f}")

print("\n10 highest IDF scores (rarest notes):")
for note, score in note_idf_sorted[-10:]:
    print(f"  {note}: {score:.2f}")

10 lowest IDF scores (most common notes):
  musk: 0.79
  bergamot: 1.03
  sandalwood: 1.08
  amber: 1.14
  jasmine: 1.14
  vanilla: 1.16
  patchouli: 1.21
  rose: 1.37
  cedar: 1.46
  vetiver: 1.78

10 highest IDF scores (rarest notes):
  white peach blossom: 10.09
  white royal lily: 10.09
  white tea blossom: 10.09
  wisteria petals: 10.09
  wolfberry: 10.09
  yellow flowers: 10.09
  yellow poppy: 10.09
  yohimbe: 10.09
  yuzu flower: 10.09
  zefir: 10.09


## Cosine similarity ##

With the weighted matrix in hand, we can use a more sophisticated similarity metric. Rather than measuring raw note overlap like Jaccard, cosine similarity measures the angle between two fragrance vectors in note space.  Cosine similarity doesn't take into account the magnitude of a vector, so the number of notes doesn't matter; two fragrances pointing in the same direction will be similar regardless of the number of notes.  

Cosine simialrity also works well with TF-IDF weighted vectors, as the weighting suppresses the impact of more common notes.

In [55]:
# cosine similarity against the target fragrance
target_vec = tfidf_matrix[target_idx]
target_norm = norm(target_vec)

scores = []
for i in range(len(tfidf_matrix)):
    if i == target_idx:
        continue
    vec = tfidf_matrix[i]
    denom = target_norm * norm(vec)
    score = np.dot(target_vec, vec) / denom if denom > 0 else 0
    scores.append((i, score))

scores.sort(key=lambda x: x[1], reverse=True)

print(f"Recommendations for: {df.iloc[target_idx]['Perfume']} by {df.iloc[target_idx]['Brand']}\n")
print(f"Its notes: {df.iloc[target_idx]['all_notes']}\n")
print("Top 10 TF-IDF + Cosine matches:")
for rank, (idx, score) in enumerate(scores[:10], 1):
    print(f"  {rank}. {df.iloc[idx]['Perfume']} by {df.iloc[idx]['Brand']} — {score:.3f}")
    print(f"     Notes: {df.iloc[idx]['all_notes']}")

Recommendations for: dakar by thera-cosmeticos

Its notes: ['spicy notes', 'tobacco leaf', 'vanilla', 'tonka bean', 'tobacco blossom', 'cacao', 'woody notes', 'dried fruits']

Top 10 TF-IDF + Cosine matches:
  1. sweetobacco by in-the-box — 1.000
     Notes: ['tobacco leaf', 'spicy notes', 'vanilla', 'tobacco blossom', 'cacao', 'tonka bean', 'woody notes', 'dried fruits']
  2. efrate by nuancielo — 1.000
     Notes: ['spicy notes', 'tobacco leaf', 'vanilla', 'cacao', 'tobacco blossom', 'tonka bean', 'dried fruits', 'woody notes']
  3. essences-44 by nuancielo — 1.000
     Notes: ['tobacco leaf', 'spicy notes', 'cacao', 'vanilla', 'tonka bean', 'tobacco blossom', 'dried fruits', 'woody notes']
  4. tobacco-vanille by tom-ford — 1.000
     Notes: ['tobacco leaf', 'spicy notes', 'vanilla', 'cacao', 'tonka bean', 'tobacco blossom', 'dried fruits', 'woody notes']
  5. vanille-persuasive by lpdo — 0.843
     Notes: ['tobacco leaf', 'spicy notes', 'vanilla', 'tonka bean', 'tobacco blossom', '

Both Jaccard and cosine similarity rank the first four in the list highly, since they're efectively the same formulations under different names.  The real meaningful differences show up further down the list, where cosine has suggested Precious Moment and Amber Oud tobacco edition.  The fragrances share fewer notes overall, but the shared notes are rarer and therefore more meaningful.

## Nonnegative Matrix Factorization ##

Cosine similarity tells us how similar two fragrances are, but it doesn't tell us anything about the underlying structure of the fragrance. Nonnegative matrix factorization (NMF) takes a different approach — instead of comparing fragrances directly, it decomposes the entire matrix to discover latent structure. Given a matrix and a number k, NMF finds k "archetypes" — groups of notes that tend to appear together — without being told what those archetypes should be.

NMF is well suited to this data because our matrix is inherently non-negative even after the weighting is applied. A fragrance either contains a note or it doesn't; there is no such thing as negative citrus.

The result is two matrices: one that maps a note to an archetype, and one that maps each fragrance to an archetype.

Michael Edwards' fragrance wheel organizes each fragrance into four major families: Fresh, Floral, Amber, and Woody. Running NMF with k=4 gives us a chance to see whether the algorithm can find those same families purely from the co occurrence of notes.

In [40]:
from sklearn.decomposition import NMF

k = 4  # number of archetypes
model = NMF(n_components=k, random_state=42, max_iter=500)
W = model.fit_transform(tfidf_matrix)  # fragrance x archetype matrix
H = model.components_                  # archetype x note matrix

print(f"W shape (fragrances x archetypes): {W.shape}")
print(f"H shape (archetypes x notes): {H.shape}")

W shape (fragrances x archetypes): (24063, 4)
H shape (archetypes x notes): (4, 1633)


In [45]:
top_n = 10

for i, component in enumerate(H):
    top_indices = component.argsort()[-top_n:][::-1]
    top_notes = [sorted_notes[j] for j in top_indices]
    print(f"Archetype {i+1}: {', '.join(top_notes)}")

Archetype 1: freesia, peony, pear, black currant, raspberry, vanilla, mandarin orange, peach, rose, musk
Archetype 2: lavender, geranium, vetiver, lemon, cardamom, cedar, mint, grapefruit, sage, nutmeg
Archetype 3: agarwood (oud), saffron, labdanum, leather, incense, cinnamon, olibanum, tobacco, guaiac wood, patchouli
Archetype 4: ylang-ylang, tuberose, carnation, aldehydes, orris root, jasmine, iris, rose, lily-of-the-valley, oakmoss


NMF found the four fragrance families quite clearly:

**Archetype 1:** Fresh - light flowers like peony and freesia and fruits like pear, blackcurrant, and peach appear in this category

**Archetype 2:** Woody - cedar and vetiver are classic woody scents

**Archetype 3:** Amber - very rich notes like oud, incense and patchouli point to the amber category

**Archetype 4:** Floral - tuberose, ylang-ylang and carnation distinguish the classic powdery florals from the fresh florals in archetype 1

Now that the four archetypes have been discovered, we can test two known fragrances - Dakar, a rich tobacco scent, and Fracas, a classic white floral.

In [56]:
# Dakar's archetype weightes
idx = 4119
weights = W[idx]

print(f"Archetype breakdown for: {df.iloc[idx]['Perfume']} by {df.iloc[idx]['Brand']}")
print(f"Its notes: {df.iloc[idx]['all_notes']}\n")

for i, weight in enumerate(weights):
    print(f"  Archetype {i+1}: {weight:.4f}")

dominant = weights.argmax() + 1
print(f"\nDominant archetype: Archetype {dominant}")

Archetype breakdown for: dakar by thera-cosmeticos
Its notes: ['spicy notes', 'tobacco leaf', 'vanilla', 'tonka bean', 'tobacco blossom', 'cacao', 'woody notes', 'dried fruits']

  Archetype 1: 0.0275
  Archetype 2: 0.0025
  Archetype 3: 0.0839
  Archetype 4: 0.0017

Dominant archetype: Archetype 3


With its spicy, tobacco notes, Dakar is clearly Archetype 3 - Amber.

In [52]:
# Fracas's archetype weightes
idx = 14210
weights = W[idx]

print(f"Archetype breakdown for: {df.iloc[idx]['Perfume']} by {df.iloc[idx]['Brand']}")
print(f"Its notes: {df.iloc[idx]['all_notes']}\n")

for i, weight in enumerate(weights):
    print(f"  Archetype {i+1}: {weight:.4f}")

dominant = weights.argmax() + 1
print(f"\nDominant archetype: Archetype {dominant}")

Archetype breakdown for: fracas by robert-piguet
Its notes: ['peach', 'orange blossom', 'hiacynth', 'green leaves', 'mandarin orange', 'bergamot', 'tuberose', 'jasmine', 'gardenia', 'osmanthus', 'narcissus', 'lily-of-the-valley', 'carnation', 'white iris', 'violet root', 'coriander', 'rose geranium', 'rose', 'musk', 'sandalwood', 'amber', 'oakmoss', 'vetiver', 'cedar']

  Archetype 1: 0.0579
  Archetype 2: 0.0524
  Archetype 3: 0.0000
  Archetype 4: 0.4701

Dominant archetype: Archetype 4


With a bouquet of floral notes, Fracas is distinctly Archetype 4 - Floral.

## End ##

Starting from a raw dataset of over 24,000 fragrances, we cleaned the notes, built a matrix, and evaluated two similarity metrics, Jaccard and cosine similarity.  The NMF section is where the notebook earned its name; nonnegative matrix factorization recovered the four fragrance families of the popular Edwards wheel from note co-occurrence alone - Fresh, Woody, Amber, and Floral.  

The next step is to use this weighted cosine similarity in a web app to serve users with a top-10 list of fragrances based on a fragrance they already like, and display the archetypes alongside them.